In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

file_path = "/Volumes/workspace/default/data_s/SM Cleaned Data BR2019.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

print("Data loaded successfully!")
print("Rows:", df.count())
print("Columns:", len(df.columns))

Data loaded successfully!
Rows: 2919315
Columns: 6


In [0]:
display(df.limit(10))

x_Timestamp,t_kWh,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz),meter
2019-07-10T00:00:00.000Z,0.021,243.1,1.79,50.02,BR02
2019-07-10T00:03:00.000Z,0.021,242.91,1.8,50.07,BR02
2019-07-10T00:06:00.000Z,0.021,242.46,1.83,50.0,BR02
2019-07-10T00:09:00.000Z,0.02,241.27,1.79,49.95,BR02
2019-07-10T00:12:00.000Z,0.02,240.77,1.79,49.98,BR02
2019-07-10T00:15:00.000Z,0.021,240.97,1.8,49.99,BR02
2019-07-10T00:18:00.000Z,0.02,241.16,1.79,49.99,BR02
2019-07-10T00:21:00.000Z,0.021,241.56,1.79,50.05,BR02
2019-07-10T00:24:00.000Z,0.02,241.64,1.79,50.06,BR02
2019-07-10T00:27:00.000Z,0.02,241.62,1.78,50.08,BR02


In [0]:
df_clean = (
    df
    .withColumnRenamed("x_Timestamp", "timestamp")
    .withColumnRenamed("t_kWh", "energy_kwh")
    .withColumnRenamed("z_Avg Voltage (Volt)", "avg_voltage")
    .withColumnRenamed("z_Avg Current (Amp)", "avg_current")
    .withColumnRenamed("y_Freq (Hz)", "frequency")
    .withColumnRenamed("meter", "meter_id")
)

display(df_clean.limit(10))

timestamp,energy_kwh,avg_voltage,avg_current,frequency,meter_id
2019-07-10T00:00:00.000Z,0.021,243.1,1.79,50.02,BR02
2019-07-10T00:03:00.000Z,0.021,242.91,1.8,50.07,BR02
2019-07-10T00:06:00.000Z,0.021,242.46,1.83,50.0,BR02
2019-07-10T00:09:00.000Z,0.02,241.27,1.79,49.95,BR02
2019-07-10T00:12:00.000Z,0.02,240.77,1.79,49.98,BR02
2019-07-10T00:15:00.000Z,0.021,240.97,1.8,49.99,BR02
2019-07-10T00:18:00.000Z,0.02,241.16,1.79,49.99,BR02
2019-07-10T00:21:00.000Z,0.021,241.56,1.79,50.05,BR02
2019-07-10T00:24:00.000Z,0.02,241.64,1.79,50.06,BR02
2019-07-10T00:27:00.000Z,0.02,241.62,1.78,50.08,BR02


In [0]:
hourly_df = (
    df_clean
    .withColumn("hour_timestamp", F.date_trunc("hour", "timestamp"))
    .groupBy("meter_id", "hour_timestamp")
    .agg(
        F.sum("energy_kwh").alias("energy_kwh"),
        F.avg("avg_voltage").alias("avg_voltage"),
        F.avg("avg_current").alias("avg_current"),
        F.avg("frequency").alias("frequency")
    )
)

display(hourly_df.limit(20))

meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency
BR02,2019-07-15T10:00:00.000Z,0.17700000000000007,252.00100000000003,0.868,50.0025
BR02,2019-08-01T13:00:00.000Z,0.24300000000000005,237.00500000000005,1.193,50.0595
BR02,2019-08-01T21:00:00.000Z,0.25800000000000006,233.21200000000005,1.3375,49.936
BR02,2019-08-09T04:00:00.000Z,0.2920000000000001,215.6105,1.4555,42.517500000000005
BR02,2019-08-11T13:00:00.000Z,0.28300000000000003,231.39999999999995,1.5480000000000003,47.465
BR02,2019-08-27T23:00:00.000Z,0.4120000000000001,230.16799999999998,1.9125,49.9
BR02,2019-08-28T11:00:00.000Z,0.3050000000000001,239.584,1.4995000000000005,50.00749999999999
BR02,2019-09-05T07:00:00.000Z,0.06200000000000002,235.07200000000003,0.5350000000000001,47.543000000000006
BR02,2019-09-15T18:00:00.000Z,0.14100000000000004,247.11400000000003,0.7565,49.936499999999995
BR02,2019-09-18T13:00:00.000Z,0.0,0.0,0.0,0.0


In [0]:
print("Original rows:", df.count())
print("Hourly rows:", hourly_df.count())
print("Columns:", hourly_df.columns)

Original rows: 2919315
Hourly rows: 145972
Columns: ['meter_id', 'hour_timestamp', 'energy_kwh', 'avg_voltage', 'avg_current', 'frequency']


In [0]:
hourly_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- hour_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- avg_voltage: double (nullable = true)
 |-- avg_current: double (nullable = true)
 |-- frequency: double (nullable = true)



In [0]:
hourly_df = (
    hourly_df
    .withColumn("hour", F.hour("hour_timestamp"))
    .withColumn("day", F.dayofmonth("hour_timestamp"))
    .withColumn("month", F.month("hour_timestamp"))
    .withColumn("year", F.year("hour_timestamp"))
    .withColumn("day_of_week", F.dayofweek("hour_timestamp"))
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("hour_timestamp").isin([1, 7]), 1).otherwise(0)
    )
)

display(hourly_df.limit(10))

meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend
BR02,2019-07-15T10:00:00.000Z,0.17700000000000007,252.00100000000003,0.868,50.0025,10,15,7,2019,2,0
BR02,2019-08-01T13:00:00.000Z,0.24300000000000005,237.00500000000005,1.193,50.0595,13,1,8,2019,5,0
BR02,2019-08-01T21:00:00.000Z,0.25800000000000006,233.21200000000005,1.3375,49.936,21,1,8,2019,5,0
BR02,2019-08-09T04:00:00.000Z,0.2920000000000001,215.6105,1.4555,42.517500000000005,4,9,8,2019,6,0
BR02,2019-08-11T13:00:00.000Z,0.28300000000000003,231.39999999999995,1.5480000000000003,47.465,13,11,8,2019,1,1
BR02,2019-08-27T23:00:00.000Z,0.4120000000000001,230.16799999999998,1.9125,49.9,23,27,8,2019,3,0
BR02,2019-08-28T11:00:00.000Z,0.3050000000000001,239.584,1.4995000000000005,50.00749999999999,11,28,8,2019,4,0
BR02,2019-09-05T07:00:00.000Z,0.06200000000000002,235.07200000000003,0.5350000000000001,47.543000000000006,7,5,9,2019,5,0
BR02,2019-09-15T18:00:00.000Z,0.14100000000000004,247.11400000000003,0.7565,49.936499999999995,18,15,9,2019,1,1
BR02,2019-09-18T13:00:00.000Z,0.0,0.0,0.0,0.0,13,18,9,2019,4,0


In [0]:
window_spec = (
    Window
    .partitionBy("meter_id")
    .orderBy("hour_timestamp")
)

hourly_df = (
    hourly_df
    .withColumn(
        "energy_lag_1",
        F.lag("energy_kwh", 1).over(window_spec)
    )
    .withColumn(
        "energy_lag_2",
        F.lag("energy_kwh", 2).over(window_spec)
    )
    .withColumn(
        "energy_lag_24",
        F.lag("energy_kwh", 24).over(window_spec)
    )
    .withColumn(
        "energy_lag_168",
        F.lag("energy_kwh", 168).over(window_spec)
    )
)

display(hourly_df.limit(20))

meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168
BR06,2019-07-11T00:00:00.000Z,0.114,77.40950000000001,0.7100000000000001,15.009500000000003,0,11,7,2019,5,0,null,null,null,null
BR06,2019-07-11T01:00:00.000Z,0.021,26.737000000000002,0.1935,5.006,1,11,7,2019,5,0,0.114,null,null,null
BR06,2019-07-11T02:00:00.000Z,0.09999999999999999,65.1115,0.6405000000000001,12.514,2,11,7,2019,5,0,0.021,0.114,null,null
BR06,2019-07-11T03:00:00.000Z,0.10300000000000001,80.78450000000001,0.639,15.016499999999999,3,11,7,2019,5,0,0.09999999999999999,0.021,null,null
BR06,2019-07-11T04:00:00.000Z,0.318,201.3635,2.0029999999999997,37.549499999999995,4,11,7,2019,5,0,0.10300000000000001,0.09999999999999999,null,null
BR06,2019-07-11T05:00:00.000Z,0.129,67.22399999999999,0.8230000000000001,12.5025,5,11,7,2019,5,0,0.318,0.10300000000000001,null,null
BR06,2019-07-11T06:00:00.000Z,0.8890000000000001,267.87149999999997,4.086999999999999,49.981,6,11,7,2019,5,0,0.129,0.318,null,null
BR06,2019-07-11T07:00:00.000Z,0.6110000000000003,268.9845,3.339,50.039500000000004,7,11,7,2019,5,0,0.8890000000000001,0.129,null,null
BR06,2019-07-11T08:00:00.000Z,0.5240000000000002,240.1715,2.4605,45.0735,8,11,7,2019,5,0,0.6110000000000003,0.8890000000000001,null,null
BR06,2019-07-11T09:00:00.000Z,0.5800000000000002,261.06199999999995,2.9004999999999996,50.05799999999999,9,11,7,2019,5,0,0.5240000000000002,0.6110000000000003,null,null


In [0]:
Window.partitionBy("meter_id").orderBy("hour_timestamp")

WindowSpec(PartitionBy(meter_id), OrderBy(hour_timestamp ASC NULLS FIRST))

In [0]:
rolling_window = (
    Window
    .partitionBy("meter_id")
    .orderBy("hour_timestamp")
    .rowsBetween(-24, -1)
)

hourly_df = hourly_df.withColumn(
    "rolling_avg_24h",
    F.avg("energy_kwh").over(rolling_window)
)

display(hourly_df.limit(30))

meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168,rolling_avg_24h
BR06,2019-07-11T00:00:00.000Z,0.114,77.40950000000001,0.7100000000000001,15.009500000000003,0,11,7,2019,5,0,null,null,null,null,null
BR06,2019-07-11T01:00:00.000Z,0.021,26.737000000000002,0.1935,5.006,1,11,7,2019,5,0,0.114,null,null,null,0.114
BR06,2019-07-11T02:00:00.000Z,0.09999999999999999,65.1115,0.6405000000000001,12.514,2,11,7,2019,5,0,0.021,0.114,null,null,0.0675
BR06,2019-07-11T03:00:00.000Z,0.10300000000000001,80.78450000000001,0.639,15.016499999999999,3,11,7,2019,5,0,0.09999999999999999,0.021,null,null,0.07833333333333332
BR06,2019-07-11T04:00:00.000Z,0.318,201.3635,2.0029999999999997,37.549499999999995,4,11,7,2019,5,0,0.10300000000000001,0.09999999999999999,null,null,0.08449999999999999
BR06,2019-07-11T05:00:00.000Z,0.129,67.22399999999999,0.8230000000000001,12.5025,5,11,7,2019,5,0,0.318,0.10300000000000001,null,null,0.13119999999999998
BR06,2019-07-11T06:00:00.000Z,0.8890000000000001,267.87149999999997,4.086999999999999,49.981,6,11,7,2019,5,0,0.129,0.318,null,null,0.13083333333333333
BR06,2019-07-11T07:00:00.000Z,0.6110000000000003,268.9845,3.339,50.039500000000004,7,11,7,2019,5,0,0.8890000000000001,0.129,null,null,0.23914285714285713
BR06,2019-07-11T08:00:00.000Z,0.5240000000000002,240.1715,2.4605,45.0735,8,11,7,2019,5,0,0.6110000000000003,0.8890000000000001,null,null,0.285625
BR06,2019-07-11T09:00:00.000Z,0.5800000000000002,261.06199999999995,2.9004999999999996,50.05799999999999,9,11,7,2019,5,0,0.5240000000000002,0.6110000000000003,null,null,0.3121111111111111


In [0]:
hourly_df = hourly_df.withColumn(
    "target_energy",
    F.lead("energy_kwh", 1).over(window_spec)
)

display(hourly_df.limit(20))

meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168,rolling_avg_24h,target_energy
BR06,2019-07-11T00:00:00.000Z,0.114,77.40950000000001,0.7100000000000001,15.009500000000003,0,11,7,2019,5,0,null,null,null,null,null,0.021
BR06,2019-07-11T01:00:00.000Z,0.021,26.737000000000002,0.1935,5.006,1,11,7,2019,5,0,0.114,null,null,null,0.114,0.09999999999999999
BR06,2019-07-11T02:00:00.000Z,0.09999999999999999,65.1115,0.6405000000000001,12.514,2,11,7,2019,5,0,0.021,0.114,null,null,0.0675,0.10300000000000001
BR06,2019-07-11T03:00:00.000Z,0.10300000000000001,80.78450000000001,0.639,15.016499999999999,3,11,7,2019,5,0,0.09999999999999999,0.021,null,null,0.07833333333333332,0.318
BR06,2019-07-11T04:00:00.000Z,0.318,201.3635,2.0029999999999997,37.549499999999995,4,11,7,2019,5,0,0.10300000000000001,0.09999999999999999,null,null,0.08449999999999999,0.129
BR06,2019-07-11T05:00:00.000Z,0.129,67.22399999999999,0.8230000000000001,12.5025,5,11,7,2019,5,0,0.318,0.10300000000000001,null,null,0.13119999999999998,0.8890000000000001
BR06,2019-07-11T06:00:00.000Z,0.8890000000000001,267.87149999999997,4.086999999999999,49.981,6,11,7,2019,5,0,0.129,0.318,null,null,0.13083333333333333,0.6110000000000003
BR06,2019-07-11T07:00:00.000Z,0.6110000000000003,268.9845,3.339,50.039500000000004,7,11,7,2019,5,0,0.8890000000000001,0.129,null,null,0.23914285714285713,0.5240000000000002
BR06,2019-07-11T08:00:00.000Z,0.5240000000000002,240.1715,2.4605,45.0735,8,11,7,2019,5,0,0.6110000000000003,0.8890000000000001,null,null,0.285625,0.5800000000000002
BR06,2019-07-11T09:00:00.000Z,0.5800000000000002,261.06199999999995,2.9004999999999996,50.05799999999999,9,11,7,2019,5,0,0.5240000000000002,0.6110000000000003,null,null,0.3121111111111111,1.5799999999999996


In [0]:
model_df = hourly_df.dropna(
    subset=[
        "energy_lag_1",
        "energy_lag_2",
        "energy_lag_24",
        "energy_lag_168",
        "rolling_avg_24h",
        "target_energy"
    ]
)

print("Rows available for ML:", model_df.count())

display(model_df.limit(10))

Rows available for ML: 138198


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168,rolling_avg_24h,target_energy
BR06,2019-07-30T00:00:00.000Z,0.8400000000000001,237.38950000000006,3.6270000000000002,50.02599999999999,0,30,7,2019,3,0,0.3760000000000001,0.724,0.28400000000000014,0.114,0.539375,0.8380000000000002
BR06,2019-07-30T01:00:00.000Z,0.8380000000000002,235.35549999999998,3.691499999999999,50.017999999999994,1,30,7,2019,3,0,0.8400000000000001,0.3760000000000001,0.27400000000000013,0.021,0.5625416666666668,0.8220000000000002
BR06,2019-07-30T02:00:00.000Z,0.8220000000000002,237.7935,3.5909999999999997,50.003,2,30,7,2019,3,0,0.8380000000000002,0.8400000000000001,0.21000000000000008,0.09999999999999999,0.5860416666666668,0.7780000000000001
BR06,2019-07-30T03:00:00.000Z,0.7780000000000001,241.28249999999997,3.3625000000000007,49.9705,3,30,7,2019,3,0,0.8220000000000002,0.8380000000000002,0.2060000000000001,0.10300000000000001,0.6115416666666669,0.8250000000000001
BR06,2019-07-30T04:00:00.000Z,0.8250000000000001,242.961,3.5985,49.975999999999985,4,30,7,2019,3,0,0.7780000000000001,0.8220000000000002,0.2080000000000001,0.318,0.6353750000000001,0.6300000000000002
BR06,2019-07-30T05:00:00.000Z,0.6300000000000002,248.38899999999998,3.1394999999999995,49.93599999999999,5,30,7,2019,3,0,0.8250000000000001,0.7780000000000001,0.2510000000000001,0.129,0.6610833333333335,0.9510000000000001
BR06,2019-07-30T06:00:00.000Z,0.9510000000000001,254.448,4.311999999999999,49.8975,6,30,7,2019,3,0,0.6300000000000002,0.8250000000000001,0.49500000000000016,0.8890000000000001,0.6768750000000002,0.28200000000000003
BR06,2019-07-30T07:00:00.000Z,0.28200000000000003,254.9,1.2655000000000005,49.99399999999999,7,30,7,2019,3,0,0.9510000000000001,0.6300000000000002,0.5600000000000002,0.6110000000000003,0.6958750000000001,0.43300000000000016
BR06,2019-07-30T08:00:00.000Z,0.43300000000000016,252.721,2.0780000000000003,50.030499999999996,8,30,7,2019,3,0,0.28200000000000003,0.9510000000000001,0.46600000000000014,0.5240000000000002,0.6842916666666667,1.035
BR06,2019-07-30T09:00:00.000Z,1.035,246.33450000000002,4.3685,49.9985,9,30,7,2019,3,0,0.43300000000000016,0.28200000000000003,1.578,0.5800000000000002,0.6829166666666668,0.5680000000000003


In [0]:
model_df = model_df.orderBy("hour_timestamp")

In [0]:
display(model_df.select(
    "meter_id",
    "hour_timestamp",
    "energy_kwh",
    "target_energy"
).limit(20))

meter_id,hour_timestamp,energy_kwh,target_energy
BR52,2019-05-16T00:00:00.000Z,0.7270000000000004,0.6760000000000004
BR52,2019-05-16T01:00:00.000Z,0.6760000000000004,0.6810000000000004
BR52,2019-05-16T02:00:00.000Z,0.6810000000000004,0.6910000000000004
BR52,2019-05-16T03:00:00.000Z,0.6910000000000004,0.6470000000000004
BR52,2019-05-16T04:00:00.000Z,0.6470000000000004,0.42500000000000016
BR52,2019-05-16T05:00:00.000Z,0.42500000000000016,0.19400000000000006
BR52,2019-05-16T06:00:00.000Z,0.19400000000000006,0.14100000000000004
BR52,2019-05-16T07:00:00.000Z,0.14100000000000004,0.41
BR52,2019-05-16T08:00:00.000Z,0.41,0.26700000000000007
BR52,2019-05-16T09:00:00.000Z,0.26700000000000007,0.20500000000000007


In [0]:
model_df.select(
    F.min("hour_timestamp").alias("start_time"),
    F.max("hour_timestamp").alias("end_time")
).show()

+-------------------+-------------------+
|         start_time|           end_time|
+-------------------+-------------------+
|2019-05-16 00:00:00|2019-12-31 22:00:00|
+-------------------+-------------------+



In [0]:
split_time = model_df.select(
    F.expr("percentile_approx(hour_timestamp, 0.8)").alias("split_time")
).collect()[0]["split_time"]

print("Split time:", split_time)

Split time: 2019-11-30 19:00:00


In [0]:
train_df = model_df.filter(
    F.col("hour_timestamp") <= F.lit(split_time)
)

test_df = model_df.filter(
    F.col("hour_timestamp") > F.lit(split_time)
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 110552
Testing rows: 27646


In [0]:
feature_cols = [
    "hour",
    "day",
    "month",
    "day_of_week",
    "is_weekend",
    "avg_voltage",
    "avg_current",
    "frequency",
    "energy_lag_1",
    "energy_lag_2",
    "energy_lag_24",
    "energy_lag_168",
    "rolling_avg_24h"
]

In [0]:
target_col = "target_energy"

In [0]:
model_df = model_df.orderBy("hour_timestamp")

display(model_df.select(
    "meter_id",
    "hour_timestamp",
    "energy_kwh",
    "target_energy"
).limit(20))

meter_id,hour_timestamp,energy_kwh,target_energy
BR52,2019-05-16T00:00:00.000Z,0.7270000000000004,0.6760000000000004
BR52,2019-05-16T01:00:00.000Z,0.6760000000000004,0.6810000000000004
BR52,2019-05-16T02:00:00.000Z,0.6810000000000004,0.6910000000000004
BR52,2019-05-16T03:00:00.000Z,0.6910000000000004,0.6470000000000004
BR52,2019-05-16T04:00:00.000Z,0.6470000000000004,0.42500000000000016
BR52,2019-05-16T05:00:00.000Z,0.42500000000000016,0.19400000000000006
BR52,2019-05-16T06:00:00.000Z,0.19400000000000006,0.14100000000000004
BR52,2019-05-16T07:00:00.000Z,0.14100000000000004,0.41
BR52,2019-05-16T08:00:00.000Z,0.41,0.26700000000000007
BR52,2019-05-16T09:00:00.000Z,0.26700000000000007,0.20500000000000007


In [0]:
split_time = model_df.select(
    F.expr("percentile_approx(hour_timestamp, 0.8)").alias("split_time")
).collect()[0]["split_time"]

print("Split time:", split_time)

Split time: 2019-11-30 19:00:00


In [0]:
train_df = model_df.filter(
    F.col("hour_timestamp") <= F.lit(split_time)
)

test_df = model_df.filter(
    F.col("hour_timestamp") > F.lit(split_time)
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 110552
Testing rows: 27646


In [0]:
%pip install xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [0]:
print("df:", "df" in globals())
print("df_clean:", "df_clean" in globals())
print("hourly_df:", "hourly_df" in globals())
print("model_df:", "model_df" in globals())
print("train_df:", "train_df" in globals())
print("test_df:", "test_df" in globals())

df: False
df_clean: False
hourly_df: False
model_df: False
train_df: False
test_df: False


In [0]:
print("df:", "df" in globals())
print("df_clean:", "df_clean" in globals())
print("hourly_df:", "hourly_df" in globals())
print("model_df:", "model_df" in globals())
print("train_df:", "train_df" in globals())
print("test_df:", "test_df" in globals())

df: False
df_clean: False
hourly_df: False
model_df: False
train_df: False
test_df: False


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

file_path = "/Volumes/workspace/default/data_s/SM Cleaned Data BR2019.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

print("Data loaded!")
print("Rows:", df.count())
print("Columns:", len(df.columns))

Data loaded!
Rows: 2919315
Columns: 6


In [0]:
df_clean = (
    df
    .withColumnRenamed("x_Timestamp", "timestamp")
    .withColumnRenamed("t_kWh", "energy_kwh")
    .withColumnRenamed("z_Avg Voltage (Volt)", "avg_voltage")
    .withColumnRenamed("z_Avg Current (Amp)", "avg_current")
    .withColumnRenamed("y_Freq (Hz)", "frequency")
    .withColumnRenamed("meter", "meter_id")
)

display(df_clean.limit(10))

timestamp,energy_kwh,avg_voltage,avg_current,frequency,meter_id
2019-07-10T00:00:00.000Z,0.021,243.1,1.79,50.02,BR02
2019-07-10T00:03:00.000Z,0.021,242.91,1.8,50.07,BR02
2019-07-10T00:06:00.000Z,0.021,242.46,1.83,50.0,BR02
2019-07-10T00:09:00.000Z,0.02,241.27,1.79,49.95,BR02
2019-07-10T00:12:00.000Z,0.02,240.77,1.79,49.98,BR02
2019-07-10T00:15:00.000Z,0.021,240.97,1.8,49.99,BR02
2019-07-10T00:18:00.000Z,0.02,241.16,1.79,49.99,BR02
2019-07-10T00:21:00.000Z,0.021,241.56,1.79,50.05,BR02
2019-07-10T00:24:00.000Z,0.02,241.64,1.79,50.06,BR02
2019-07-10T00:27:00.000Z,0.02,241.62,1.78,50.08,BR02


In [0]:
hourly_df = (
    df_clean
    .withColumn("hour_timestamp", F.date_trunc("hour", "timestamp"))
    .groupBy("meter_id", "hour_timestamp")
    .agg(
        F.sum("energy_kwh").alias("energy_kwh"),
        F.avg("avg_voltage").alias("avg_voltage"),
        F.avg("avg_current").alias("avg_current"),
        F.avg("frequency").alias("frequency")
    )
)

print("Hourly rows:", hourly_df.count())

Hourly rows: 145972


In [0]:
hourly_df = (
    hourly_df
    .withColumn("hour", F.hour("hour_timestamp"))
    .withColumn("day", F.dayofmonth("hour_timestamp"))
    .withColumn("month", F.month("hour_timestamp"))
    .withColumn("year", F.year("hour_timestamp"))
    .withColumn("day_of_week", F.dayofweek("hour_timestamp"))
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("hour_timestamp").isin([1, 7]), 1).otherwise(0)
    )
)

In [0]:
window_spec = (
    Window
    .partitionBy("meter_id")
    .orderBy("hour_timestamp")
)

hourly_df = (
    hourly_df
    .withColumn("energy_lag_1", F.lag("energy_kwh", 1).over(window_spec))
    .withColumn("energy_lag_2", F.lag("energy_kwh", 2).over(window_spec))
    .withColumn("energy_lag_24", F.lag("energy_kwh", 24).over(window_spec))
    .withColumn("energy_lag_168", F.lag("energy_kwh", 168).over(window_spec))
)

In [0]:
rolling_window = (
    Window
    .partitionBy("meter_id")
    .orderBy("hour_timestamp")
    .rowsBetween(-24, -1)
)

hourly_df = hourly_df.withColumn(
    "rolling_avg_24h",
    F.avg("energy_kwh").over(rolling_window)
)

In [0]:
hourly_df = hourly_df.withColumn(
    "target_energy",
    F.lead("energy_kwh", 1).over(window_spec)
)

In [0]:
model_df = hourly_df.dropna(
    subset=[
        "energy_lag_1",
        "energy_lag_2",
        "energy_lag_24",
        "energy_lag_168",
        "rolling_avg_24h",
        "target_energy"
    ]
)

print("Rows available for ML:", model_df.count())

Rows available for ML: 138198


In [0]:
model_df = model_df.orderBy("hour_timestamp")

split_time = model_df.select(
    F.expr("percentile_approx(hour_timestamp, 0.8)").alias("split_time")
).collect()[0]["split_time"]

print("Split time:", split_time)

Split time: 2019-11-30 19:00:00


In [0]:
train_df = model_df.filter(
    F.col("hour_timestamp") <= F.lit(split_time)
)

test_df = model_df.filter(
    F.col("hour_timestamp") > F.lit(split_time)
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 110552
Testing rows: 27646


In [0]:
print("df:", df.count())
print("hourly_df:", hourly_df.count())
print("model_df:", model_df.count())
print("train_df:", train_df.count())
print("test_df:", test_df.count())

df: 2919315
hourly_df: 145972
model_df: 138198
train_df: 110552
test_df: 27646


In [0]:
feature_cols = [
    "hour",
    "day",
    "month",
    "day_of_week",
    "is_weekend",
    "avg_voltage",
    "avg_current",
    "frequency",
    "energy_lag_1",
    "energy_lag_2",
    "energy_lag_24",
    "energy_lag_168",
    "rolling_avg_24h"
]

target_col = "target_energy"

print("Number of features:", len(feature_cols))
print("Target:", target_col)

Number of features: 13
Target: target_energy


In [0]:
train_pd = train_df.select(
    feature_cols + [target_col]
).toPandas()

test_pd = test_df.select(
    feature_cols + [target_col]
).toPandas()

print("Training shape:", train_pd.shape)
print("Testing shape:", test_pd.shape)

Training shape: (110552, 14)
Testing shape: (27646, 14)


In [0]:
X_train = train_pd[feature_cols]
y_train = train_pd[target_col]

X_test = test_pd[feature_cols]
y_test = test_pd[target_col]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (110552, 13)
y_train: (110552,)
X_test: (27646, 13)
y_test: (27646,)


In [0]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("XGBoost training completed!")

XGBoost training completed!


In [0]:
y_pred_xgb = xgb_model.predict(X_test)

print("Prediction completed!")
print("Number of predictions:", len(y_pred_xgb))

Prediction completed!
Number of predictions: 27646


In [0]:
import pandas as pd

results_xgb = pd.DataFrame({
    "actual_energy": y_test.values,
    "predicted_energy": y_pred_xgb
})

display(results_xgb.head(20))

actual_energy,predicted_energy
0.8290000000000002,0.72445124
0.23000000000000007,0.2277876
0.007,0.0678888
0.004,0.021077298
0.4320000000000001,0.4205495
0.3080000000000001,0.29044735
0.3380000000000001,0.41078117
0.031000000000000014,0.03255529
0.09000000000000001,0.05569022
0.18800000000000003,0.11498948


In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost Results")
print("----------------")
print("MAE :", mae_xgb)
print("RMSE:", rmse_xgb)
print("R²  :", r2_xgb)

XGBoost Results
----------------
MAE : 0.10554093954914533
RMSE: 0.22030256792208777
R²  : 0.58549320096299


In [0]:
feature_cols = [
    "hour",
    "day",
    "month",
    "day_of_week",
    "is_weekend",
    "avg_voltage",
    "avg_current",
    "frequency",
    "energy_lag_1",
    "energy_lag_2",
    "energy_lag_24",
    "energy_lag_168",
    "rolling_avg_24h"
]

target_col = "target_energy"

print("Feature and target columns are ready!")

Feature and target columns are ready!


In [0]:
print("train_df rows:", train_df.count())
print("test_df rows:", test_df.count())

train_df rows: 110552
test_df rows: 27646


In [0]:
X_train = train_pd[feature_cols]
y_train = train_pd[target_col]

X_test = test_pd[feature_cols]
y_test = test_pd[target_col]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (110552, 13)
y_train: (110552,)
X_test: (27646, 13)
y_test: (27646,)


In [0]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed!")

Random Forest training completed!


In [0]:
y_pred_rf = rf_model.predict(X_test)

print("Random Forest prediction completed!")
print("Number of predictions:", len(y_pred_rf))

Random Forest prediction completed!
Number of predictions: 27646


In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results")
print("---------------------")
print("MAE :", mae_rf)
print("RMSE:", rmse_rf)
print("R²  :", r2_rf)

Random Forest Results
---------------------
MAE : 0.10642746727148075
RMSE: 0.21936413829317175
R²  : 0.5890170543487513


In [0]:
anomaly_features = [
    "energy_kwh",
    "avg_voltage",
    "avg_current",
    "frequency"
]

anomaly_pd = model_df.select(
    anomaly_features
).toPandas()

print("Anomaly data shape:", anomaly_pd.shape)

Anomaly data shape: (138198, 4)


In [0]:
from sklearn.ensemble import IsolationForest

iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

iso_model.fit(anomaly_pd[anomaly_features])

print("Isolation Forest training completed!")

Isolation Forest training completed!


In [0]:
anomaly_pd["anomaly"] = iso_model.predict(
    anomaly_pd[anomaly_features]
)

print("Anomaly detection completed!")

Anomaly detection completed!


In [0]:
print("Normal readings:", (anomaly_pd["anomaly"] == 1).sum())
print("Anomalous readings:", (anomaly_pd["anomaly"] == -1).sum())

Normal readings: 136816
Anomalous readings: 1382


In [0]:
anomalies = anomaly_pd[
    anomaly_pd["anomaly"] == -1
]

print("Total anomalies:", len(anomalies))

display(anomalies.head(20))

Total anomalies: 1382


energy_kwh,avg_voltage,avg_current,frequency,anomaly
0.607,112.468,3.4445,27.509000000000004,-1
0.40900000000000003,70.035,2.3195000000000006,14.976999999999999,-1
0.035,19.629,0.346,5.001,-1
2.0460000000000003,226.35299999999998,10.895500000000002,50.013,-1
1.9430000000000003,228.506,10.547000000000002,49.9935,-1
2.24,237.49249999999998,12.192,49.863,-1
2.037,226.98050000000003,9.188999999999998,49.94449999999999,-1
0.939,150.279,5.129500000000001,32.4415,-1
0.22400000000000006,0.0,0.0,0.0,-1
1.011,168.11049999999997,5.9445,37.4985,-1


In [0]:
# Get timestamp and meter ID from model_df
anomaly_info_pd = model_df.select(
    "hour_timestamp",
    "meter_id",
    "energy_kwh",
    "avg_voltage",
    "avg_current",
    "frequency"
).toPandas()

# Add Isolation Forest result
anomaly_info_pd["anomaly"] = iso_model.predict(
    anomaly_info_pd[anomaly_features]
)

# Keep only anomalous readings
anomalies_final = anomaly_info_pd[
    anomaly_info_pd["anomaly"] == -1
]

print("Total anomalies:", len(anomalies_final))

display(anomalies_final.head(20))

Total anomalies: 1382


hour_timestamp,meter_id,energy_kwh,avg_voltage,avg_current,frequency,anomaly
2019-06-13T15:00:00.000Z,BR52,0.607,112.468,3.4445,27.509000000000004,-1
2019-06-21T16:00:00.000Z,BR52,0.40900000000000003,70.035,2.3195000000000006,14.976999999999999,-1
2019-06-27T14:00:00.000Z,BR52,0.035,19.629,0.346,5.001,-1
2019-07-23T00:00:00.000Z,BR18,2.0460000000000003,226.35299999999998,10.895500000000002,50.013,-1
2019-07-23T01:00:00.000Z,BR18,1.9430000000000003,228.506,10.547000000000002,49.9935,-1
2019-07-23T06:00:00.000Z,BR18,2.24,237.49249999999998,12.192,49.863,-1
2019-07-23T14:00:00.000Z,BR13,2.037,226.98050000000003,9.188999999999998,49.94449999999999,-1
2019-07-23T14:00:00.000Z,BR18,0.939,150.279,5.129500000000001,32.4415,-1
2019-07-23T16:00:00.000Z,BR17,0.22400000000000006,0.0,0.0,0.0,-1
2019-07-23T17:00:00.000Z,BR18,1.011,168.11049999999997,5.9445,37.4985,-1


In [0]:
from pyspark.sql import functions as F

monthly_usage = (
    model_df
    .withColumn("month", F.date_format("hour_timestamp", "yyyy-MM"))
    .groupBy("meter_id", "month")
    .agg(
        F.sum("energy_kwh").alias("units_consumed")
    )
    .orderBy("meter_id", "month")
)

display(monthly_usage)

meter_id,month,units_consumed
BR02,2019-07,27.110000000000024
BR02,2019-08,172.55000000000004
BR02,2019-09,145.42500000000007
BR02,2019-10,117.67100000000018
BR02,2019-11,89.97100000000002
BR02,2019-12,99.119
BR03,2019-07,43.61800000000002
BR03,2019-08,291.8460000000002
BR03,2019-09,236.9500000000001
BR03,2019-10,142.17499999999998


In [0]:
from pyspark.sql import functions as F

bill_df = (
    monthly_usage
    .withColumn(
        "energy_charge",
        F.when(
            F.col("units_consumed") <= 50,
            F.col("units_consumed") * 3
        )
        .when(
            F.col("units_consumed") <= 150,
            50 * 3 + (F.col("units_consumed") - 50) * 5
        )
        .when(
            F.col("units_consumed") <= 250,
            50 * 3 + 100 * 5 +
            (F.col("units_consumed") - 150) * 7
        )
        .otherwise(
            50 * 3 + 100 * 5 + 100 * 7 +
            (F.col("units_consumed") - 250) * 10
        )
    )
)

display(bill_df)

meter_id,month,units_consumed,energy_charge
BR02,2019-07,27.110000000000024,81.33000000000007
BR02,2019-08,172.55000000000004,807.8500000000003
BR02,2019-09,145.42500000000007,627.1250000000003
BR02,2019-10,117.67100000000018,488.35500000000087
BR02,2019-11,89.97100000000002,349.8550000000001
BR02,2019-12,99.119,395.595
BR03,2019-07,43.61800000000002,130.85400000000007
BR03,2019-08,291.8460000000002,1768.4600000000019
BR03,2019-09,236.9500000000001,1258.6500000000008
BR03,2019-10,142.17499999999998,610.8749999999999


In [0]:
user_settings = spark.createDataFrame(
    [
        ("BR03", 200.0, 1500.0),
        ("BR02", 150.0, 1200.0),
        ("BR04", 250.0, 1800.0)
    ],
    ["meter_id", "monthly_target_kwh", "monthly_budget"]
)

display(user_settings)

meter_id,monthly_target_kwh,monthly_budget
BR03,200.0,1500.0
BR02,150.0,1200.0
BR04,250.0,1800.0


In [0]:
budget_df = (
    bill_df
    .join(
        user_settings,
        on="meter_id",
        how="inner"
    )
    .withColumn(
        "usage_difference",
        F.col("units_consumed") - F.col("monthly_target_kwh")
    )
    .withColumn(
        "budget_difference",
        F.col("energy_charge") - F.col("monthly_budget")
    )
)

display(budget_df)

meter_id,month,units_consumed,energy_charge,monthly_target_kwh,monthly_budget,usage_difference,budget_difference
BR02,2019-07,27.110000000000024,81.33000000000007,150.0,1200.0,-122.88999999999997,-1118.6699999999998
BR03,2019-07,43.61800000000002,130.85400000000007,200.0,1500.0,-156.38199999999998,-1369.146
BR04,2019-07,40.04300000000003,120.12900000000008,250.0,1800.0,-209.95699999999997,-1679.8709999999999
BR02,2019-08,172.55000000000004,807.8500000000003,150.0,1200.0,22.55000000000004,-392.14999999999975
BR03,2019-08,291.8460000000002,1768.4600000000019,200.0,1500.0,91.84600000000017,268.46000000000186
BR04,2019-08,399.2400000000004,2842.400000000004,250.0,1800.0,149.2400000000004,1042.4000000000042
BR02,2019-09,145.42500000000007,627.1250000000003,150.0,1200.0,-4.574999999999932,-572.8749999999997
BR03,2019-09,236.9500000000001,1258.6500000000008,200.0,1500.0,36.9500000000001,-241.34999999999923
BR04,2019-09,309.38100000000014,1943.8100000000013,250.0,1800.0,59.38100000000014,143.8100000000013
BR02,2019-10,117.67100000000018,488.35500000000087,150.0,1200.0,-32.32899999999982,-711.6449999999991


In [0]:
budget_status_df = (
    budget_df
    .withColumn(
        "usage_status",
        F.when(
            F.col("units_consumed") <= F.col("monthly_target_kwh"),
            "WITHIN TARGET"
        ).otherwise("TARGET EXCEEDED")
    )
    .withColumn(
        "budget_status",
        F.when(
            F.col("energy_charge") <= F.col("monthly_budget"),
            "WITHIN BUDGET"
        ).otherwise("BUDGET EXCEEDED")
    )
)

display(
    budget_status_df.select(
        "meter_id",
        "month",
        "units_consumed",
        "monthly_target_kwh",
        "usage_status",
        "energy_charge",
        "monthly_budget",
        "budget_status"
    )
)

meter_id,month,units_consumed,monthly_target_kwh,usage_status,energy_charge,monthly_budget,budget_status
BR02,2019-07,27.110000000000024,150.0,WITHIN TARGET,81.33000000000007,1200.0,WITHIN BUDGET
BR03,2019-07,43.61800000000002,200.0,WITHIN TARGET,130.85400000000007,1500.0,WITHIN BUDGET
BR04,2019-07,40.04300000000003,250.0,WITHIN TARGET,120.12900000000008,1800.0,WITHIN BUDGET
BR02,2019-08,172.55000000000004,150.0,TARGET EXCEEDED,807.8500000000003,1200.0,WITHIN BUDGET
BR03,2019-08,291.8460000000002,200.0,TARGET EXCEEDED,1768.4600000000019,1500.0,BUDGET EXCEEDED
BR04,2019-08,399.2400000000004,250.0,TARGET EXCEEDED,2842.400000000004,1800.0,BUDGET EXCEEDED
BR02,2019-09,145.42500000000007,150.0,WITHIN TARGET,627.1250000000003,1200.0,WITHIN BUDGET
BR03,2019-09,236.9500000000001,200.0,TARGET EXCEEDED,1258.6500000000008,1500.0,WITHIN BUDGET
BR04,2019-09,309.38100000000014,250.0,TARGET EXCEEDED,1943.8100000000013,1800.0,BUDGET EXCEEDED
BR02,2019-10,117.67100000000018,150.0,WITHIN TARGET,488.35500000000087,1200.0,WITHIN BUDGET


In [0]:
budget_guardian_df = (
    budget_status_df
    .withColumn(
        "avg_daily_usage",
        F.col("units_consumed") /
        F.dayofmonth(
            F.last_day(
                F.to_date(F.concat(F.col("month"), F.lit("-01")))
            )
        )
    )
)

display(
    budget_guardian_df.select(
        "meter_id",
        "month",
        "units_consumed",
        "monthly_target_kwh",
        "avg_daily_usage",
        "usage_status",
        "energy_charge",
        "monthly_budget",
        "budget_status"
    )
)

meter_id,month,units_consumed,monthly_target_kwh,avg_daily_usage,usage_status,energy_charge,monthly_budget,budget_status
BR02,2019-07,27.110000000000024,150.0,0.8745161290322588,WITHIN TARGET,81.33000000000007,1200.0,WITHIN BUDGET
BR03,2019-07,43.61800000000002,200.0,1.407032258064517,WITHIN TARGET,130.85400000000007,1500.0,WITHIN BUDGET
BR04,2019-07,40.04300000000003,250.0,1.2917096774193557,WITHIN TARGET,120.12900000000008,1800.0,WITHIN BUDGET
BR02,2019-08,172.55000000000004,150.0,5.566129032258066,TARGET EXCEEDED,807.8500000000003,1200.0,WITHIN BUDGET
BR03,2019-08,291.8460000000002,200.0,9.414387096774199,TARGET EXCEEDED,1768.4600000000019,1500.0,BUDGET EXCEEDED
BR04,2019-08,399.2400000000004,250.0,12.878709677419367,TARGET EXCEEDED,2842.400000000004,1800.0,BUDGET EXCEEDED
BR02,2019-09,145.42500000000007,150.0,4.847500000000002,WITHIN TARGET,627.1250000000003,1200.0,WITHIN BUDGET
BR03,2019-09,236.9500000000001,200.0,7.898333333333337,TARGET EXCEEDED,1258.6500000000008,1500.0,WITHIN BUDGET
BR04,2019-09,309.38100000000014,250.0,10.312700000000005,TARGET EXCEEDED,1943.8100000000013,1800.0,BUDGET EXCEEDED
BR02,2019-10,117.67100000000018,150.0,3.795838709677425,WITHIN TARGET,488.35500000000087,1200.0,WITHIN BUDGET


In [0]:
budget_guardian_df = (
    budget_guardian_df
    .withColumn(
        "days_in_month",
        F.dayofmonth(
            F.last_day(
                F.to_date(
                    F.concat(F.col("month"), F.lit("-01"))
                )
            )
        )
    )
    .withColumn(
        "projected_monthly_usage",
        F.col("avg_daily_usage") * F.col("days_in_month")
    )
)

display(
    budget_guardian_df.select(
        "meter_id",
        "month",
        "units_consumed",
        "avg_daily_usage",
        "days_in_month",
        "projected_monthly_usage",
        "monthly_target_kwh",
        "usage_status"
    )
)

meter_id,month,units_consumed,avg_daily_usage,days_in_month,projected_monthly_usage,monthly_target_kwh,usage_status
BR02,2019-07,27.110000000000024,0.8745161290322588,31,27.110000000000024,150.0,WITHIN TARGET
BR03,2019-07,43.61800000000002,1.407032258064517,31,43.61800000000002,200.0,WITHIN TARGET
BR04,2019-07,40.04300000000003,1.2917096774193557,31,40.04300000000003,250.0,WITHIN TARGET
BR02,2019-08,172.55000000000004,5.566129032258066,31,172.55000000000004,150.0,TARGET EXCEEDED
BR03,2019-08,291.8460000000002,9.414387096774199,31,291.8460000000002,200.0,TARGET EXCEEDED
BR04,2019-08,399.2400000000004,12.878709677419367,31,399.2400000000004,250.0,TARGET EXCEEDED
BR02,2019-09,145.42500000000007,4.847500000000002,30,145.42500000000007,150.0,WITHIN TARGET
BR03,2019-09,236.9500000000001,7.898333333333337,30,236.9500000000001,200.0,TARGET EXCEEDED
BR04,2019-09,309.38100000000014,10.312700000000005,30,309.38100000000014,250.0,TARGET EXCEEDED
BR02,2019-10,117.67100000000018,3.795838709677425,31,117.67100000000018,150.0,WITHIN TARGET


In [0]:
budget_guardian_df = (
    budget_guardian_df
    .withColumn(
        "projected_usage_difference",
        F.col("projected_monthly_usage") - F.col("monthly_target_kwh")
    )
    .withColumn(
        "projected_bill",
        F.when(
            F.col("projected_monthly_usage") <= 50,
            F.col("projected_monthly_usage") * 3
        )
        .when(
            F.col("projected_monthly_usage") <= 150,
            50 * 3 +
            (F.col("projected_monthly_usage") - 50) * 5
        )
        .when(
            F.col("projected_monthly_usage") <= 250,
            50 * 3 +
            100 * 5 +
            (F.col("projected_monthly_usage") - 150) * 7
        )
        .otherwise(
            50 * 3 +
            100 * 5 +
            100 * 7 +
            (F.col("projected_monthly_usage") - 250) * 10
        )
    )
    .withColumn(
        "projected_budget_difference",
        F.col("projected_bill") - F.col("monthly_budget")
    )
)

display(
    budget_guardian_df.select(
        "meter_id",
        "month",
        "units_consumed",
        "projected_monthly_usage",
        "monthly_target_kwh",
        "projected_usage_difference",
        "projected_bill",
        "monthly_budget",
        "projected_budget_difference"
    )
)

meter_id,month,units_consumed,projected_monthly_usage,monthly_target_kwh,projected_usage_difference,projected_bill,monthly_budget,projected_budget_difference
BR02,2019-07,27.110000000000024,27.110000000000024,150.0,-122.88999999999997,81.33000000000007,1200.0,-1118.6699999999998
BR03,2019-07,43.61800000000002,43.61800000000002,200.0,-156.38199999999998,130.85400000000007,1500.0,-1369.146
BR04,2019-07,40.04300000000003,40.04300000000003,250.0,-209.95699999999997,120.12900000000008,1800.0,-1679.8709999999999
BR02,2019-08,172.55000000000004,172.55000000000004,150.0,22.55000000000004,807.8500000000003,1200.0,-392.14999999999975
BR03,2019-08,291.8460000000002,291.8460000000002,200.0,91.84600000000017,1768.4600000000019,1500.0,268.46000000000186
BR04,2019-08,399.2400000000004,399.2400000000004,250.0,149.2400000000004,2842.400000000004,1800.0,1042.4000000000042
BR02,2019-09,145.42500000000007,145.42500000000007,150.0,-4.574999999999932,627.1250000000003,1200.0,-572.8749999999997
BR03,2019-09,236.9500000000001,236.9500000000001,200.0,36.9500000000001,1258.6500000000008,1500.0,-241.34999999999923
BR04,2019-09,309.38100000000014,309.38100000000014,250.0,59.38100000000014,1943.8100000000013,1800.0,143.8100000000013
BR02,2019-10,117.67100000000018,117.67100000000018,150.0,-32.32899999999982,488.35500000000087,1200.0,-711.6449999999991


In [0]:
budget_guardian_df = (
    budget_guardian_df
    .withColumn(
        "guardian_status",
        F.when(
            (F.col("projected_usage_difference") <= 0) &
            (F.col("projected_budget_difference") <= 0),
            "ON TRACK"
        )
        .when(
            (F.col("projected_usage_difference") > 0) &
            (F.col("projected_budget_difference") > 0),
            "USAGE AND BUDGET RISK"
        )
        .when(
            F.col("projected_usage_difference") > 0,
            "USAGE TARGET RISK"
        )
        .otherwise(
            "BUDGET RISK"
        )
    )
)

display(
    budget_guardian_df.select(
        "meter_id",
        "month",
        "units_consumed",
        "monthly_target_kwh",
        "projected_monthly_usage",
        "projected_bill",
        "monthly_budget",
        "guardian_status"
    )
)

meter_id,month,units_consumed,monthly_target_kwh,projected_monthly_usage,projected_bill,monthly_budget,guardian_status
BR02,2019-07,27.110000000000024,150.0,27.110000000000024,81.33000000000007,1200.0,ON TRACK
BR03,2019-07,43.61800000000002,200.0,43.61800000000002,130.85400000000007,1500.0,ON TRACK
BR04,2019-07,40.04300000000003,250.0,40.04300000000003,120.12900000000008,1800.0,ON TRACK
BR02,2019-08,172.55000000000004,150.0,172.55000000000004,807.8500000000003,1200.0,USAGE TARGET RISK
BR03,2019-08,291.8460000000002,200.0,291.8460000000002,1768.4600000000019,1500.0,USAGE AND BUDGET RISK
BR04,2019-08,399.2400000000004,250.0,399.2400000000004,2842.400000000004,1800.0,USAGE AND BUDGET RISK
BR02,2019-09,145.42500000000007,150.0,145.42500000000007,627.1250000000003,1200.0,ON TRACK
BR03,2019-09,236.9500000000001,200.0,236.9500000000001,1258.6500000000008,1500.0,USAGE TARGET RISK
BR04,2019-09,309.38100000000014,250.0,309.38100000000014,1943.8100000000013,1800.0,USAGE AND BUDGET RISK
BR02,2019-10,117.67100000000018,150.0,117.67100000000018,488.35500000000087,1200.0,ON TRACK


In [0]:
budget_guardian_df = budget_guardian_df.withColumn(
    "recommended_daily_usage",
    F.col("monthly_target_kwh") / F.col("days_in_month")
)

display(
    budget_guardian_df.select(
        "meter_id",
        "month",
        "units_consumed",
        "monthly_target_kwh",
        "avg_daily_usage",
        "recommended_daily_usage",
        "guardian_status"
    )
)

meter_id,month,units_consumed,monthly_target_kwh,avg_daily_usage,recommended_daily_usage,guardian_status
BR02,2019-07,27.110000000000024,150.0,0.8745161290322588,4.838709677419355,ON TRACK
BR03,2019-07,43.61800000000002,200.0,1.407032258064517,6.451612903225806,ON TRACK
BR04,2019-07,40.04300000000003,250.0,1.2917096774193557,8.064516129032258,ON TRACK
BR02,2019-08,172.55000000000004,150.0,5.566129032258066,4.838709677419355,USAGE TARGET RISK
BR03,2019-08,291.8460000000002,200.0,9.414387096774199,6.451612903225806,USAGE AND BUDGET RISK
BR04,2019-08,399.2400000000004,250.0,12.878709677419367,8.064516129032258,USAGE AND BUDGET RISK
BR02,2019-09,145.42500000000007,150.0,4.847500000000002,5.0,ON TRACK
BR03,2019-09,236.9500000000001,200.0,7.898333333333337,6.666666666666667,USAGE TARGET RISK
BR04,2019-09,309.38100000000014,250.0,10.312700000000005,8.333333333333334,USAGE AND BUDGET RISK
BR02,2019-10,117.67100000000018,150.0,3.795838709677425,4.838709677419355,ON TRACK
